# TemporalWiki Cloze Eval Pipeline

This notebook builds an example of the **evaluation infrastructure** for the *Contradiction-Aware Sparse Memory Finetuning (CASF)* continual learning project.

The core idea is simple: Wikidata triples are structured facts of the form `(subject, relation, object)`. By hiding the object, we can turn any triple into a cloze prompt that tests whether a language model knows that fact:

## Setup

In [4]:
# uv pip install pandas
import zipfile
import pandas as pd
from collections import Counter
import re
import os

DIFFSETS_ZIP = "data/TWiki_Diffsets.zip"
PROBES_ZIP   = "data/TWiki_Probes.zip"

assert os.path.exists(DIFFSETS_ZIP), f"Missing {DIFFSETS_ZIP}"
assert os.path.exists(PROBES_ZIP),   f"Missing {PROBES_ZIP}"
print("Both zip files found.")

Both zip files found.


## 1. Load Diffsets (Training Corpus)

The diffsets are raw Wikipedia text passages that changed between consecutive monthly snapshots (Aug–Dec 2020).  
These are the **training signal** — finetuning on them is what drives knowledge updates in the model.

> **Data notes:** Each snapshot has ~800–900k passages (median ~1,300 chars), ~35k duplicates, and a meaningful number of very short passages that are likely section headers or noise. **Deduplicate and filter short passages before finetuning.**


In [5]:
diffset_files = {
    "0809": "TWiki_Diffsets/wikipedia_0809_gpt2.csv",
    "0910": "TWiki_Diffsets/wikipedia_0910_gpt2.csv",
    "1011": "TWiki_Diffsets/wikipedia_1011_gpt2.csv",
    "1112": "TWiki_Diffsets/wikipedia_1112_gpt2.csv",
}

diffsets = {}
with zipfile.ZipFile(DIFFSETS_ZIP, "r") as z:
    for key, path in diffset_files.items():
        with z.open(path) as f:
            diffsets[key] = pd.read_csv(f)

for key, df in diffsets.items():
    lengths = df["text"].str.len()
    dupes   = df.duplicated().sum()
    short   = (lengths < 100).sum()
    print(f"Diffset {key}: {len(df):,} passages | "
          f"median {lengths.median():.0f} chars | "
          f"dupes={dupes:,} | short(<100 chars)={short:,}")

Diffset 0809: 808,867 passages | median 1311 chars | dupes=35,533 | short(<100 chars)=78,967
Diffset 0910: 806,485 passages | median 1329 chars | dupes=35,851 | short(<100 chars)=79,203
Diffset 1011: 809,383 passages | median 1314 chars | dupes=34,700 | short(<100 chars)=86,518
Diffset 1112: 896,218 passages | median 1257 chars | dupes=34,420 | short(<100 chars)=131,401


## 2. Load Probes (Eval Signal)

Each probe is a `(subject, relation, object)` Wikidata triple. Per snapshot, there are two splits:

- `changed` — facts updated in that snapshot → **plasticity test**
- `unchanged` — facts that stayed the same → **stability test**


In [6]:
probe_files = {
    "0801-0901": {"changed": "twiki_probes/0801-0901_changed.csv",   "unchanged": "twiki_probes/0801-0901_unchanged.csv"},
    "0901-1001": {"changed": "twiki_probes/0901-1001_changed.csv",   "unchanged": "twiki_probes/0901-1001_unchanged.csv"},
    "1001-1101": {"changed": "twiki_probes/1001-1101_changed.csv",   "unchanged": "twiki_probes/1001-1101_unchanged.csv"},
    "1101-1201": {"changed": "twiki_probes/1101-1201_changed.csv",   "unchanged": "twiki_probes/1101-1201_unchanged.csv"},
}

probes = {}
with zipfile.ZipFile(PROBES_ZIP, "r") as z:
    for period, splits in probe_files.items():
        probes[period] = {}
        for split, path in splits.items():
            with z.open(path) as f:
                probes[period][split] = pd.read_csv(f)

print(f"{'Period':<15} {'Changed':>10} {'Unchanged':>12} {'% Changed':>12}")
print("-" * 52)
for period, splits in probes.items():
    c = len(splits["changed"])
    u = len(splits["unchanged"])
    print(f"{period:<15} {c:>10,} {u:>12,} {100*c/(c+u):>11.1f}%")

Period             Changed    Unchanged    % Changed
----------------------------------------------------
0801-0901            1,776        6,935        20.4%
0901-1001            1,982        7,340        21.3%
1001-1101            1,358        7,313        15.7%
1101-1201            1,951        7,293        21.1%


## 3. Probe Structure

Let's inspect a sample of triples from each split — this informs how we write verbalization templates.


In [7]:
for split in ["changed", "unchanged"]:
    print(f"\n--- {split.upper()} probes (0801-0901) ---")
    print(probes["0801-0901"][split].head(10).to_string(index=False))


--- CHANGED probes (0801-0901) ---
             subject               relation               object
   Nagpur Metro Rail               has part       Congress Nagar
     Come and Get It               composer       Paul McCartney
     Kool & The Gang               has part          Ronald Bell
Turnberry Lighthouse                  color                white
   instant messaging            subclass of             software
      Folsom Library                part of Rensselaer Libraries
       Medtronic plc               replaces          Omar Ishrak
             Taliban            chairperson       Akhtar Mansour
              Queens   determination method               census
     Mangkunegara IX country of citizenship            Indonesia

--- UNCHANGED probes (0801-0901) ---
                 subject              relation           object
                    Zola           instance of         business
               Henry VII        place of birth  Pembroke Castle
     Flamingoes in 

In [29]:
# Unique relations across all changed probes
all_changed = pd.concat([probes[p]["changed"] for p in probes], ignore_index=True)
print(f"Total changed probe triples: {len(all_changed):,}")
print(f"Unique relations in changed: {all_changed['relation'].nunique()}")
print(f"\nTop 15 relations (changed probes):")
print(all_changed["relation"].value_counts().head(15).to_string())

Total changed probe triples: 7,067
Unique relations in changed: 351

Top 15 relations (changed probes):
relation
occupation                                          294
languages spoken written or signed                  289
place of birth                                      261
instance of                                         238
place of death                                      226
country of citizenship                              196
country                                             196
located in the administrative territorial entity    168
language of work or name                            167
educated at                                         143
has part                                            140
given name                                          135
country for sport                                   126
native language                                     118
different from                                      114


## 4. Verbalization: Triples → Cloze Prompts

Each `(subject, relation)` pair is mapped to a natural language prompt. The model is expected to complete it with the `object`.

> **Tip:** expand `RELATION_TEMPLATES` to cover more relations and improve eval coverage.


In [22]:
RELATION_TEMPLATES = {

    # ── People ────────────────────────────────────────────────────────────
    "occupation":                                       "The occupation of {subject} is",
    "sex or gender":                                    "The sex or gender of {subject} is",
    "place of birth":                                   "{subject} was born in",
    "place of death":                                   "{subject} died in",
    "cause of death":                                   "The cause of death of {subject} is",
    "place of burial":                                  "{subject} was buried in",
    "country of citizenship":                           "{subject} is a citizen of",
    "residence":                                        "The residence of {subject} is",
    "educated at":                                      "{subject} was educated at",
    "academic degree":                                  "The academic degree of {subject} is",
    "academic major":                                   "The academic major of {subject} is",
    "employer":                                         "{subject} is employed by",
    "work location":                                    "The work location of {subject} is",
    "work period (start)":                              "{subject} started working in",
    "work period (end)":                                "{subject} stopped working in",
    "position held":                                    "The position held by {subject} is",
    "member of political party":                        "{subject} is a member of the political party",
    "parliamentary group":                              "The parliamentary group of {subject} is",
    "military rank":                                    "The military rank of {subject} is",
    "military branch":                                  "The military branch of {subject} is",
    "award received":                                   "{subject} received the award",
    "nominated for":                                    "{subject} was nominated for",
    "notable work":                                     "A notable work of {subject} is",
    "given name":                                       "The given name of {subject} is",
    "family name":                                      "The family name of {subject} is",
    "father":                                           "The father of {subject} is",
    "mother":                                           "The mother of {subject} is",
    "sibling":                                          "{subject} is a sibling of",
    "child":                                            "The child of {subject} is",
    "spouse":                                           "The spouse of {subject} is",
    "languages spoken written or signed":               "The language spoken by {subject} is",
    "native language":                                  "The native language of {subject} is",
    "country for sport":                                "The country {subject} represents in sport is",
    "member of sports team":                            "{subject} is a member of sports team",
    "position played on team / speciality":             "The position played by {subject} is",
    "sports discipline competed in":                    "{subject} competed in the sports discipline",
    "significant event":                                "A significant event in the life of {subject} is",

    # ── Geography ─────────────────────────────────────────────────────────
    "country":                                          "The country of {subject} is",
    "country of origin":                                "The country of origin of {subject} is",
    "location":                                         "The location of {subject} is",
    "located in the administrative territorial entity": "{subject} is located in",
    "contains administrative territorial entity":       "{subject} contains the administrative territorial entity",
    "shares border with":                               "{subject} shares a border with",
    "capital":                                          "The capital of {subject} is",
    "historic county":                                  "The historic county of {subject} is",
    "electoral district":                               "The electoral district of {subject} is",

    # ── Organizations ─────────────────────────────────────────────────────
    "chairperson":                                      "The chairperson of {subject} is",
    "founded by":                                       "{subject} was founded by",
    "headquarters location":                            "The headquarters of {subject} is located in",
    "owned by":                                         "{subject} is owned by",
    "operator":                                         "The operator of {subject} is",
    "member of":                                        "{subject} is a member of",
    "replaces":                                         "{subject} replaces",
    "replaced by":                                      "{subject} was replaced by",
    "part of":                                          "{subject} is part of",
    "has part":                                         "{subject} has part",
    "religion":                                         "The religion of {subject} is",

    # ── Works & Media ─────────────────────────────────────────────────────
    "author":                                           "The author of {subject} is",
    "composer":                                         "The composer of {subject} is",
    "director":                                         "The director of {subject} is",
    "screenwriter":                                     "The screenwriter of {subject} is",
    "producer":                                         "The producer of {subject} is",
    "performer":                                        "The performer of {subject} is",
    "cast member":                                      "A cast member of {subject} is",
    "characters":                                       "A character in {subject} is",
    "character role":                                   "The character role of {subject} is",
    "creator":                                          "The creator of {subject} is",
    "manufacturer":                                     "The manufacturer of {subject} is",
    "publisher":                                        "The publisher of {subject} is",
    "distributed by":                                   "{subject} is distributed by",
    "record label":                                     "The record label of {subject} is",
    "platform":                                         "The platform of {subject} is",
    "original broadcaster":                             "The original broadcaster of {subject} is",
    "original language of film or TV show":             "The original language of {subject} is",
    "language of work or name":                         "The language of {subject} is",
    "writing language":                                 "The writing language of {subject} is",
    "publication date":                                 "{subject} was published in",
    "genre":                                            "The genre of {subject} is",
    "form of creative work":                            "{subject} is a form of",
    "narrative location":                               "The narrative location of {subject} is",
    "set in environment":                               "{subject} is set in the environment of",
    "main subject":                                     "The main subject of {subject} is",
    "has works in the collection":                      "{subject} has works in the collection",
    "named after":                                      "{subject} is named after",
    "follows":                                          "{subject} follows",
    "followed by":                                      "{subject} is followed by",
    "winner":                                           "The winner of {subject} is",

    # ── Sports ────────────────────────────────────────────────────────────
    "league":                                           "The league of {subject} is",
    "conflict":                                         "{subject} participated in the conflict",

    # ── Science & Nature ──────────────────────────────────────────────────
    "taxon rank":                                       "The taxon rank of {subject} is",
    "parent taxon":                                     "The parent taxon of {subject} is",
    "taxon author":                                     "The taxon author of {subject} is",
    "instrument":                                       "The instrument played by {subject} is",

    # ── Identity & Classification ─────────────────────────────────────────
    "instance of":                                      "{subject} is an instance of",
    "subclass of":                                      "{subject} is a subclass of",
    "different from":                                   "{subject} is different from",
    "field of work":                                    "The field of work of {subject} is",
    "color":                                            "The color of {subject} is",
    "determination method":                             "The determination method of {subject} is",
    "sport":                                            "The sport of {subject} is",
    "participant in":                                   "{subject} participated in",

    # ── Wikidata Structural (noisy — consider excluding from metrics) ──────
    "stated in":                                        "{subject} is stated in",
    "inferred from":                                    "{subject} is inferred from",
    "acquisition transaction":                          "The acquisition transaction of {subject} is",
    "subject has role":                                 "The role of {subject} is",
    "object has role":                                  "The role of the object in {subject} is",
    "applies to part":                                  "{subject} applies to part",
    "for work":                                         "{subject} is for the work",
    "of":                                               "{subject} is of",
}


def verbalize(subject: str, relation: str) -> str | None:
    """Convert (subject, relation) to a cloze prompt. Returns None if no template exists."""
    template = RELATION_TEMPLATES.get(relation)
    if template is None:
        return None
    return template.format(subject=subject)


# Quick test
examples = [
    ("Taliban",         "chairperson"),
    ("Henry VII",       "place of birth"),
    ("Chorzyna",        "country"),
    ("Come and Get It", "composer"),
    ("some entity",     "some unknown relation"),
]
for subj, rel in examples:
    print(f"  [{rel}]  →  {verbalize(subj, rel)!r}")

  [chairperson]  →  'The chairperson of Taliban is'
  [place of birth]  →  'Henry VII was born in'
  [country]  →  'The country of Chorzyna is'
  [composer]  →  'The composer of Come and Get It is'
  [some unknown relation]  →  None


In [30]:
# Coverage audit across all probe triples
all_probes = pd.concat(
    [probes[p][s] for p in probes for s in ["changed", "unchanged"]],
    ignore_index=True
)
all_probes["prompt"] = all_probes.apply(
    lambda r: verbalize(r["subject"], r["relation"]), axis=1
)

covered   = all_probes["prompt"].notna().sum()
uncovered = all_probes["prompt"].isna().sum()
total     = len(all_probes)
print(f"Total probe triples:  {total:,}")
print(f"Covered by template:  {covered:,}  ({100*covered/total:.1f}%)")
print(f"Missing template:     {uncovered:,}  ({100*uncovered/total:.1f}%)")

if uncovered > 0:
    print("\nTop uncovered relations (we can later add templates for these):")
    missing = all_probes[all_probes["prompt"].isna()]["relation"].value_counts().head(15)
    print(missing.to_string())

Total probe triples:  35,948
Covered by template:  31,947  (88.9%)
Missing template:     4,001  (11.1%)

Top uncovered relations (we can later add templates for these):
relation
student of                             52
connecting line                        52
parent organization                    51
product or material produced           50
made from material                     50
location of formation                  49
mountain range                         47
place of publication                   47
developer                              46
participant                            45
located in or next to body of water    45
mouth of the watercourse               45
architect                              44
unmarried partner                      43
use                                    41


## 5. Scoring Function

Three levels of match between a model's prediction and the ground truth `object`:

| Metric | Description |
|--------|-------------|
| **Exact match** | Normalized lowercase equality — strict |
| **Contains match** | Ground truth appears anywhere in prediction — handles verbose completions (**primary metric**) |
| **Token F1** | Bag-of-words overlap — soft partial credit |


In [25]:
def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def exact_match(prediction: str, ground_truth: str) -> bool:
    return normalize(prediction) == normalize(ground_truth)

def contains_match(prediction: str, ground_truth: str) -> bool:
    return normalize(ground_truth) in normalize(prediction)

def token_f1(prediction: str, ground_truth: str) -> float:
    pred_tokens  = normalize(prediction).split()
    truth_tokens = normalize(ground_truth).split()
    if not pred_tokens or not truth_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

def score(prediction: str, ground_truth: str) -> dict:
    return {
        "exact":    exact_match(prediction, ground_truth),
        "contains": contains_match(prediction, ground_truth),
        "f1":       round(token_f1(prediction, ground_truth), 4),
    }

# Test cases
test_cases = [
    ("Akhtar Mansour",                "Akhtar Mansour"),   # exact
    ("The answer is Akhtar Mansour.", "Akhtar Mansour"),   # contains
    ("Mansour Akhtar",                "Akhtar Mansour"),   # partial
    ("Completely wrong answer",       "Akhtar Mansour"),   # wrong
    ("United States of America",      "United States"),    # contains
]
print(f"{'Prediction':<40} {'GT':<25} {'Exact':>6} {'Contains':>9} {'F1':>6}")
print("-" * 92)
for pred, gt in test_cases:
    s = score(pred, gt)
    print(f"{pred:<40} {gt:<25} {str(s['exact']):>6} {str(s['contains']):>9} {s['f1']:>6.3f}")

Prediction                               GT                         Exact  Contains     F1
--------------------------------------------------------------------------------------------
Akhtar Mansour                           Akhtar Mansour              True      True  1.000
The answer is Akhtar Mansour.            Akhtar Mansour             False      True  0.571
Mansour Akhtar                           Akhtar Mansour             False     False  1.000
Completely wrong answer                  Akhtar Mansour             False     False  0.000
United States of America                 United States              False      True  0.667


## 6. Full Eval Pipeline

Given any `model_generate_fn(prompt) -> str`, runs all probes for a period and returns plasticity and stability scores.


In [26]:
def evaluate_period(model_generate_fn, period: str, probes: dict, verbose: bool = False) -> dict:
    """
    Evaluate a model on all probes for a given snapshot period.

    Args:
        model_generate_fn : callable(prompt: str) -> str
        period            : one of "0801-0901", "0901-1001", "1001-1101", "1101-1201"
        probes            : the loaded probes dict
        verbose           : if True, print wrong answers

    Returns:
        dict with plasticity_n, plasticity_contains, plasticity_f1,
                   stability_n,  stability_contains,  stability_f1
    """
    results = {}
    for split in ["changed", "unchanged"]:
        df = probes[period][split].copy()
        df["prompt"] = df.apply(lambda r: verbalize(r["subject"], r["relation"]), axis=1)
        df = df[df["prompt"].notna()].reset_index(drop=True)

        contains_scores, f1_scores = [], []
        for _, row in df.iterrows():
            prediction = model_generate_fn(row["prompt"])
            s = score(prediction, str(row["object"]))
            contains_scores.append(s["contains"])
            f1_scores.append(s["f1"])
            if verbose and not s["contains"]:
                print(f"  WRONG | prompt: {row['prompt']!r}")
                print(f"         pred:   {prediction!r}")
                print(f"         truth:  {row['object']!r}")

        results[split] = {
            "n":        len(df),
            "contains": sum(contains_scores) / len(contains_scores) if contains_scores else 0.0,
            "f1":       sum(f1_scores)       / len(f1_scores)       if f1_scores       else 0.0,
        }

    label_map = {"changed": "plasticity", "unchanged": "stability"}
    return {
        f"{label_map[s]}_{metric}": results[s][metric]
        for s in ["changed", "unchanged"]
        for metric in ["n", "contains", "f1"]
    }


def evaluate_all_periods(model_generate_fn, probes: dict) -> pd.DataFrame:
    """Run evaluate_period across all snapshots and return a summary DataFrame."""
    rows = []
    for period in probes:
        r = evaluate_period(model_generate_fn, period, probes)
        rows.append({"period": period, **r})
    return pd.DataFrame(rows)

In [27]:
# Smoke test with a dummy model (replace with real generate() for baseline eval)
def dummy_model(prompt: str) -> str:
    """Placeholder — always returns empty string."""
    return ""

results_df = evaluate_all_periods(dummy_model, probes)
print(results_df.to_string(index=False))

   period  plasticity_n  plasticity_contains  plasticity_f1  stability_n  stability_contains  stability_f1
0801-0901          1516              0.00066            0.0         6412                 0.0           0.0
0901-1001          1668              0.00000            0.0         6619                 0.0           0.0
1001-1101          1036              0.00000            0.0         6640                 0.0           0.0
1101-1201          1452              0.00000            0.0         6604                 0.0           0.0
